In [1]:
# imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [3]:
# load dataset & check target

df = pd.read_csv("nhanes_cardiometabolic_subset_dataset.csv")

target = "HYPERTENSION_YN"

print("Dataset shape:", df.shape)

print("\nTarget missing values:")
print(df[target].isnull().sum())

print("\nTarget distribution:")
print(df[target].value_counts())

print("\nTarget proportions:")
print(df[target].value_counts(normalize=True).round(3))

Dataset shape: (10175, 64)

Target missing values:
0

Target distribution:
HYPERTENSION_YN
0    7115
1    3060
Name: count, dtype: int64

Target proportions:
HYPERTENSION_YN
0    0.699
1    0.301
Name: proportion, dtype: float64


In [5]:
# drop bp variables + id

drop_cols = [
    "SEQN",
    "BPXSY1","BPXSY2","BPXSY3","BPXSY4",
    "BPXDI1","BPXDI2","BPXDI3","BPXDI4",
    "BPXSY_AVG","BPXDI_AVG"
]

drop_cols = [col for col in drop_cols if col in df.columns]

df_model = df.drop(columns=drop_cols)

print("Shape after dropping BP + ID:", df_model.shape)

Shape after dropping BP + ID: (10175, 53)


In [7]:
# drop high missingness variables (>50%)

missing_percent = df_model.isnull().mean() * 100

high_missing_cols = missing_percent[missing_percent > 50].index.tolist()

print("Columns to drop (>50% missing):")
print(high_missing_cols)

df_model = df_model.drop(columns=high_missing_cols)

print("\nNew shape:", df_model.shape)

Columns to drop (>50% missing):
['BPQ030', 'BPQ050A', 'LBXIN', 'DIQ070', 'LBDLDL', 'LBXTR', 'RATIO_TG_HDL', 'URXUCR', 'ACR_MG_PER_G', 'SMQ040', 'ALQ120Q', 'ALQ120U', 'PAD615', 'DBQ010']

New shape: (10175, 39)


In [9]:
# define x and y

X = df_model.drop(columns=[target])
y = df_model[target]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (10175, 38)
Target shape: (10175,)


In [11]:
# drop leakage variables

leakage_cols = ["BP_CATEGORY", "HTN_STAGE"]

leakage_cols = [col for col in leakage_cols if col in X.columns]

X = X.drop(columns=leakage_cols)

print("Shape after dropping leakage variables:", X.shape)

Shape after dropping leakage variables: (10175, 36)


In [13]:
# train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True).round(3))

Training shape: (8140, 36)
Test shape: (2035, 36)

Train distribution:
HYPERTENSION_YN
0    0.699
1    0.301
Name: proportion, dtype: float64

Test distribution:
HYPERTENSION_YN
0    0.699
1    0.301
Name: proportion, dtype: float64


In [15]:
# preprocessing -- impute & scale

from sklearn.compose import ColumnTransformer

numeric_features = X_train.columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features)
    ]
)

print("Preprocessing ready.")

Preprocessing ready.


In [17]:
# test different k values

k_values = [1, 3, 5, 7, 9, 15, 25]
knn_results = []

for k in k_values:
    knn_model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", KNeighborsClassifier(n_neighbors=k))
    ])
    
    knn_model.fit(X_train, y_train)
    y_pred = knn_model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    knn_results.append(acc)
    
    print(f"K = {k} | Accuracy = {acc:.3f}")

K = 1 | Accuracy = 0.864
K = 3 | Accuracy = 0.884
K = 5 | Accuracy = 0.886
K = 7 | Accuracy = 0.893
K = 9 | Accuracy = 0.891
K = 15 | Accuracy = 0.888
K = 25 | Accuracy = 0.890


In [19]:
# pick best k

best_k = k_values[knn_results.index(max(knn_results))]
print("Best K:", best_k)

Best K: 7


In [21]:
# final knn model + full evaluation

final_knn = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=best_k))
])

final_knn.fit(X_train, y_train)
y_pred = final_knn.predict(X_test)

print("KNN Results (K=7)")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall: {recall_score(y_test, y_pred):.3f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.3f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

KNN Results (K=7)
Accuracy: 0.893
Precision: 0.892
Recall: 0.732
F1 Score: 0.804

Confusion Matrix:
[[1369   54]
 [ 164  448]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.96      0.93      1423
           1       0.89      0.73      0.80       612

    accuracy                           0.89      2035
   macro avg       0.89      0.85      0.87      2035
weighted avg       0.89      0.89      0.89      2035



In [23]:
# prepare data for clustering

cluster_df = df_model.copy()

# Drop target 
cluster_df = cluster_df.drop(columns=[target])

print("Clustering data shape:", cluster_df.shape)

Clustering data shape: (10175, 38)


In [25]:
# remove leakage variables

cluster_df = cluster_df.drop(columns=["BP_CATEGORY", "HTN_STAGE"], errors="ignore")

print("Clustering data shape after removing leakage variables:", cluster_df.shape)

Clustering data shape after removing leakage variables: (10175, 36)


In [27]:
# impute + scale

imputer = SimpleImputer(strategy="median")
cluster_imputed = imputer.fit_transform(cluster_df)

scaler = StandardScaler()
cluster_scaled = scaler.fit_transform(cluster_imputed)

print("Clustering preprocessing complete.")

Clustering preprocessing complete.


In [29]:
# try different cluster sizes (k)

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k_values = range(2, 9)

inertias = []
sil_scores = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(cluster_scaled)
    
    inertias.append(kmeans.inertia_)
    sil = silhouette_score(cluster_scaled, labels)
    sil_scores.append(sil)
    
    print(f"K = {k} | Inertia = {kmeans.inertia_:.2f} | Silhouette = {sil:.3f}")

K = 2 | Inertia = 317285.04 | Silhouette = 0.142
K = 3 | Inertia = 296438.22 | Silhouette = 0.097
K = 4 | Inertia = 284875.25 | Silhouette = 0.102
K = 5 | Inertia = 275128.73 | Silhouette = 0.101
K = 6 | Inertia = 265832.51 | Silhouette = 0.108
K = 7 | Inertia = 258947.68 | Silhouette = 0.103
K = 8 | Inertia = 253399.11 | Silhouette = 0.107


In [39]:
# final kmeans model

final_k = 2

kmeans = KMeans(n_clusters=final_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(cluster_scaled)

df["Cluster"] = clusters

print("Cluster counts:")
print(df["Cluster"].value_counts())

Cluster counts:
Cluster
1    5610
0    4565
Name: count, dtype: int64


In [40]:
# cluster summaries

cluster_summary = df.groupby("Cluster").mean(numeric_only=True).round(2)

display(cluster_summary)

,SEQN,BPXSY1,BPXSY2,BPXSY3,BPXSY4,BPXDI1,BPXDI2,BPXDI3,BPXDI4,BPXPULS,...,PAQ605,PAQ620,PAD615,PAQ650,DBQ010,DBQ700,SLQ050,SLQ060,HSD010,BPQ080
Cluster,,,,,,,,,,,,,,,,,,,,,
0,78617.48,105.74,105.35,105.50,104.72,57.19,56.78,56.24,60.14,1.00,...,1.95,1.76,114.21,1.53,1.28,2.77,1.92,2.03,2.34,1.97
1,78665.58,123.63,123.86,123.46,132.38,69.58,68.94,68.87,71.86,1.02,...,1.81,1.66,193.33,1.76,2.00,2.99,1.73,1.91,2.89,1.68
